# 05 — Chroma Vector Retrieval over Agentic RG 271 Chunks

## Purpose

This notebook builds the retrieval layer for the complaint intelligence platform.

The goal is to:
- Load agentic RG 271 chunks
- Convert each chunk into embeddings
- Store the embeddings in a local Chroma vector database
- Run semantic search using redacted complaint narratives
- Retrieve the most relevant RG 271 chunks for complaint handling context

At this stage, the focus is retrieval quality, not final LLM answer generation.

In [2]:
import pandas as pd
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 300)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
CHUNKS_PATH = Path("../data/knowledge_base/agentic_rg271_chunks.csv")

chunks_df = pd.read_csv(CHUNKS_PATH)

print("Rows:", chunks_df.shape[0])
print("Columns:", chunks_df.shape[1])

chunks_df.head()

Rows: 221
Columns: 15


,title,rg_refs,chunk_type,text,batch_id,agentic_chunk_id,source,method,text_length,has_title,has_text,has_rg_refs,too_short,too_long,passes_basic_qa
0,Dispute resolution system requirements for financial firms,['RG 271.1'],requirement,"Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act), s47(1)(h...",0,batch_0_chunk_0,ASIC RG 271,agentic_rg_aware,678,True,True,True,False,False,True
1,Obligation to comply with IDR procedures,['RG 271.2'],requirement,"Most financial firms also have a requirement to comply with their IDR procedures: see modified s912A(1)(g) and 1017G(1) of the Corporations Act, and modified s47(1)(h) and (i) of the National Credit Act.",0,batch_0_chunk_1,ASIC RG 271,agentic_rg_aware,203,True,True,True,False,False,True
2,Modified regulatory regime for unlicensed credit firms,['RG 271.3'],requirement,"A modified regulatory regime applies to some unlicensed credit firms. Credit representatives and exempt special purpose funding entities (exempt SPFEs) (including securitisation bodies) do not have IDR obligations, but must be a member of AFCA. Unlicensed carried over instrument lenders (unlicen...",0,batch_0_chunk_2,ASIC RG 271,agentic_rg_aware,688,True,True,True,False,False,True
3,Dispute resolution requirements for Australian financial services (AFS) licensees,['RG 271.4'],requirement,"Australian financial services (AFS) licensees are businesses carrying out financial services, including providing financial product advice, dealing in financial products, making a market for financial products, operating registered schemes, providing custodial or depository services, or traditio...",1,batch_1_chunk_0,ASIC RG 271,agentic_rg_aware,744,True,True,True,False,False,True
4,Dispute resolution requirements for unlicensed product issuers and unlicensed secondary sellers,['RG 271.4'],requirement,Unlicensed product issuers are issuers of financial products who are not AFS licensees. Unlicensed secondary sellers offer secondary sales of financial products under s1012C(5)(b) or (8) of the Corporations Act and are not AFS licensees. They are required to have a dispute resolution system cons...,1,batch_1_chunk_1,ASIC RG 271,agentic_rg_aware,565,True,True,True,False,False,True


In [4]:
retrieval_df = chunks_df.dropna(subset=["text"]).copy()

retrieval_df["text"] = retrieval_df["text"].astype(str)

print("Retrieval chunks:", len(retrieval_df))

retrieval_df[["title", "rg_refs", "chunk_type", "text_length", "passes_basic_qa"]].head()

Retrieval chunks: 221


,title,rg_refs,chunk_type,text_length,passes_basic_qa
0,Dispute resolution system requirements for financial firms,['RG 271.1'],requirement,678,True
1,Obligation to comply with IDR procedures,['RG 271.2'],requirement,203,True
2,Modified regulatory regime for unlicensed credit firms,['RG 271.3'],requirement,688,True
3,Dispute resolution requirements for Australian financial services (AFS) licensees,['RG 271.4'],requirement,744,True
4,Dispute resolution requirements for unlicensed product issuers and unlicensed secondary sellers,['RG 271.4'],requirement,565,True


In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [6]:
CHROMA_PATH = "../data/knowledge_base/chroma_rg271"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name="rg271_agentic_chunks"
)

print("Chroma collection ready.")
print("Collection count:", collection.count())

Chroma collection ready.
Collection count: 221


In [7]:
documents = retrieval_df["text"].tolist()

ids = [
    f"rg271_chunk_{i}"
    for i in retrieval_df.index
]

metadatas = []

for _, row in retrieval_df.iterrows():
    metadatas.append({
        "title": str(row.get("title", "")),
        "rg_refs": str(row.get("rg_refs", "")),
        "chunk_type": str(row.get("chunk_type", "")),
        "text_length": int(row.get("text_length", len(str(row.get("text", ""))))),
        "passes_basic_qa": str(row.get("passes_basic_qa", ""))
    })

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    normalize_embeddings=True
).tolist()

print("Embeddings created:", len(embeddings))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings created: 221


In [8]:
# Avoid duplicate insert if notebook is run multiple times
existing_count = collection.count()

if existing_count == 0:
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )
    print("Chunks added to Chroma.")
else:
    print("Collection already has data. Skipping add.")

print("Collection count:", collection.count())

Collection already has data. Skipping add.
Collection count: 221


In [9]:
COMPLAINTS_PATH = Path("../data/processed/cfpb_product_classification_redacted_sample.csv")

complaints_df = pd.read_csv(COMPLAINTS_PATH)

print("Rows:", complaints_df.shape[0])
print("Columns:", complaints_df.shape[1])

complaints_df.head()

Rows: 984
Columns: 4


,text,redacted_text,label,was_redacted
0,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that raise further concerns rather than resolving the dispu...","This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that raise further concerns rather than resolving the disput...",Debt collection,True
1,"XXXX XXXX, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the refund records, the funds were sent via ACH/direct d...","<PERSON>, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the refund records, the funds were sent via ACH/direct de...",Checking or savings account,True
2,I received a call from XXXX XXXX impersonating an officer of the court with legal XXXX XXXX XXXX documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. It was someone using scare tactics to get informat...,I received a call from <PERSON> impersonating an officer of the court with legal <PERSON> documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. It was someone using scare tactics to get information fro...,Debt collection,True
3,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Concern, I am writing to formally request reconsider...","<PERSON> XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Concern, I am writing to formally request reconsidera...",Checking or savings account,True
4,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. I reiterated the rules with XXXX XXXX XXXX to me b...","XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. I reiterated the rules with <PERSON> XXXX to me br...",Checking or savings account,True


In [10]:
sample_complaint = complaints_df.loc[0, "redacted_text"]

print(sample_complaint[:1500])

This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .

After my original CFPB complaint, the collector responded with additional materials that raise further concerns rather than resolving the dispute.

Among other issues : The collectors own documents appear to reflect mailing to <PERSON> rather than my actual mailing designation of <PERSON>, raising substantial concerns regarding lawful notice and delivery.

The collector continues attempting to collect large storage-related charges that escalated far beyond the apparent value of the vehicle, despite prior attempts to reasonably resolve the matter.

The collector has failed to provide competent documentation establishing a commercially reasonable lien sale process, including meaningful documentation regarding sale procedures, advertising, bidding, or disposition methodology.

The collector also failed to provide contemporaneous storage reco

In [11]:
query_embedding = embedding_model.encode(
    sample_complaint,
    normalize_embeddings=True
).tolist()

print("Query embedding created.")

Query embedding created.


In [12]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [13]:
retrieved_rows = []

for i in range(len(results["ids"][0])):
    retrieved_rows.append({
        "rank": i + 1,
        "id": results["ids"][0][i],
        "distance": results["distances"][0][i],
        "title": results["metadatas"][0][i].get("title"),
        "rg_refs": results["metadatas"][0][i].get("rg_refs"),
        "chunk_type": results["metadatas"][0][i].get("chunk_type"),
        "text": results["documents"][0][i]
    })

retrieved_df = pd.DataFrame(retrieved_rows)

retrieved_df[["rank", "distance", "title", "rg_refs", "chunk_type"]]

,rank,distance,title,rg_refs,chunk_type
0,1,0.823642,Clarification of what is not a complaint,['RG 271.33'],definition
1,2,0.887537,Complaints Involving Hardship Notices and Enforcement Postponement,['RG 271.116'],requirement
2,3,0.904746,Restrictions on enforcement actions during complaint handling,['RG 271.89'],requirement
3,4,0.946206,Expressions of dissatisfaction must not be misclassified,['RG 271.31'],requirement
4,5,0.970096,Complaints involving hardship and default notices,['RG 271.101'],other


In [14]:
for _, row in retrieved_df.iterrows():
    print(f"\n--- Result {row['rank']} ---")
    print("Distance:", row["distance"])
    print("Title:", row["title"])
    print("RG refs:", row["rg_refs"])
    print("Type:", row["chunk_type"])
    print(row["text"][:1200])


--- Result 1 ---
Distance: 0.8236422538757324
Title: Clarification of what is not a complaint
RG refs: ['RG 271.33']
Type: definition
RG 271.33 For avoidance of doubt, we do not consider the following to be ‘complaints’: (a) employment-related complaints raised by financial firm staff; (b) comments made about a firm where a response is not expected, such as: (i) feedback provided in surveys; or (ii) reports intended solely to bring a matter to a financial firm’s attention—for example, that an automatic teller machine (ATM) is damaged; (c) hardship notices or requests to postpone enforcement proceedings, unless the customer raises issues that meet the definition of complaint; and (d) reports of unauthorised transactions under the ePayments Code and disputed transactions under a chargeback process. However, we consider that a complaint has been made if the consumer raises separate issues related to the transaction that meet the definition of a complaint, or expresses dissatisfaction wit

In [15]:
manual_query = """
Customer complaint about disputed debt collection, disputed charges, 
internal dispute resolution, complaint handling obligations, and response requirements.
"""

query_embedding = embedding_model.encode(
    manual_query,
    normalize_embeddings=True
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

retrieved_rows = []

for i in range(len(results["ids"][0])):
    retrieved_rows.append({
        "rank": i + 1,
        "id": results["ids"][0][i],
        "distance": results["distances"][0][i],
        "title": results["metadatas"][0][i].get("title"),
        "rg_refs": results["metadatas"][0][i].get("rg_refs"),
        "chunk_type": results["metadatas"][0][i].get("chunk_type"),
        "text": results["documents"][0][i]
    })

manual_retrieved_df = pd.DataFrame(retrieved_rows)

manual_retrieved_df[["rank", "distance", "title", "rg_refs", "chunk_type"]]

,rank,distance,title,rg_refs,chunk_type
0,1,0.629687,Clarification of what is not a complaint,['RG 271.33'],definition
1,2,0.717296,Authorities and financial delegations for complaint resolution,['RG 271.147'],requirement
2,3,0.721575,Proactive approach to identifying complaints,['RG 271.30'],requirement
3,4,0.728399,Range of Possible Remedies for Complaints,['RG 271.161'],requirement
4,5,0.731010,Contents of a firm's complaints policy,['RG 271.173'],requirement


## Multi-Query Retrieval Test

A single raw complaint narrative can contain too many signals and may produce noisy retrieval results.

To improve retrieval quality, this section tests multiple focused retrieval queries for the same complaint:
- complaint definition
- IDR process and response requirements
- timeframes
- dispute / debt collection context

The retrieved chunks are then combined and deduplicated for review.

In [16]:
multi_queries = [
    {
        "query_name": "complaint_definition",
        "query_text": """
        Regulatory guidance on what counts as a complaint, expression of dissatisfaction,
        complaint definition, complainant, disputed transaction, and complaint handling.
        """
    },
    {
        "query_name": "idr_process_response",
        "query_text": """
        Regulatory guidance on internal dispute resolution process, IDR response,
        complaint handling obligations, written response, reasons for decision,
        and financial firm responsibilities.
        """
    },
    {
        "query_name": "idr_timeframes",
        "query_text": """
        Regulatory guidance on maximum IDR timeframes, acknowledgement of complaint,
        response time limits, complaint resolution timeframe, AFCA referral,
        and delay in complaint handling.
        """
    },
    {
        "query_name": "debt_collection_dispute",
        "query_text": """
        Consumer complaint about disputed debt collection, disputed charges,
        collection activity, financial hardship, enforcement, and internal dispute resolution.
        """
    }
]

print("Number of retrieval queries:", len(multi_queries))

Number of retrieval queries: 4


In [17]:
def run_chroma_query(query_text: str, query_name: str, n_results: int = 5) -> pd.DataFrame:
    """
    Run one semantic query against the RG 271 Chroma collection.
    """
    query_embedding = embedding_model.encode(
        query_text,
        normalize_embeddings=True
    ).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    
    rows = []
    
    for i in range(len(results["ids"][0])):
        rows.append({
            "query_name": query_name,
            "rank_within_query": i + 1,
            "id": results["ids"][0][i],
            "distance": results["distances"][0][i],
            "title": results["metadatas"][0][i].get("title"),
            "rg_refs": results["metadatas"][0][i].get("rg_refs"),
            "chunk_type": results["metadatas"][0][i].get("chunk_type"),
            "text": results["documents"][0][i]
        })
    
    return pd.DataFrame(rows)


print("Query function ready.")

Query function ready.


In [18]:
all_retrieval_results = []

for item in multi_queries:
    result_df = run_chroma_query(
        query_text=item["query_text"],
        query_name=item["query_name"],
        n_results=5
    )
    all_retrieval_results.append(result_df)

multi_retrieval_df = pd.concat(all_retrieval_results, ignore_index=True)

multi_retrieval_df[[
    "query_name",
    "rank_within_query",
    "distance",
    "title",
    "rg_refs",
    "chunk_type"
]]

,query_name,rank_within_query,distance,title,rg_refs,chunk_type
0,complaint_definition,1,0.452279,Definition of ‘complaint’ from AS/NZS 10002:2014,['RG 271.27'],definition
1,complaint_definition,2,0.649740,Trigger for obligation to handle complaints,"['RG 271.35', 'RG 271.27']",requirement
2,complaint_definition,3,0.705478,Expressions of dissatisfaction must not be misclassified,['RG 271.31'],requirement
3,complaint_definition,4,0.719399,Triggering of IDR Obligations,"['RG 271.106', 'RG 271.27', 'RG 271.32']",requirement
4,complaint_definition,5,0.736546,Determining Complaint Resolution to Complainant’s Satisfaction,['RG 271.73'],requirement
5,idr_process_response,1,0.454883,Content requirements for internal complaint management procedure,"['RG 271.177', 'RG 271.27', 'RG 271.52', 'RG 271.170', 'RG 271.101', 'RG 271.165']",requirement
6,idr_process_response,2,0.557616,Complaint management documentation requirements,['RG 271.172'],requirement
7,idr_process_response,3,0.567427,Internal complaint management procedure,"['RG 271.175', 'RG 271.176']",requirement
8,idr_process_response,4,0.590179,Training for Complaint Handling Staff,['RG 271.150'],requirement
9,idr_process_response,5,0.591080,Transition period for IDR reforms,['RG 271.25'],requirement


In [19]:
deduped_retrieval_df = (
    multi_retrieval_df
    .sort_values("distance", ascending=True)
    .groupby("id", as_index=False)
    .agg({
        "distance": "min",
        "query_name": lambda x: ", ".join(sorted(set(x))),
        "title": "first",
        "rg_refs": "first",
        "chunk_type": "first",
        "text": "first"
    })
    .sort_values("distance", ascending=True)
    .reset_index(drop=True)
)

deduped_retrieval_df["final_rank"] = deduped_retrieval_df.index + 1

deduped_retrieval_df[[
    "final_rank",
    "distance",
    "query_name",
    "title",
    "rg_refs",
    "chunk_type"
]].head(10)

,final_rank,distance,query_name,title,rg_refs,chunk_type
0,1,0.359196,idr_timeframes,Overview of maximum IDR timeframes and response requirements,"['RG 271.49', 'RG 271.50']",overview
1,2,0.401798,idr_timeframes,Purpose of enforcement action restrictions and reasonable time period,['RG 271.90'],requirement
2,3,0.408973,idr_timeframes,Contents of the Guide Regarding IDR and AFCA Interaction,"['RG 271.22', 'RG 271.106']",overview
3,4,0.423653,idr_timeframes,Contents of a firm's complaints policy,['RG 271.173'],requirement
4,5,0.428798,idr_timeframes,Factors affecting complaint response times,['RG 271.63'],other
5,6,0.452279,complaint_definition,Definition of ‘complaint’ from AS/NZS 10002:2014,['RG 271.27'],definition
6,7,0.454883,idr_process_response,Content requirements for internal complaint management procedure,"['RG 271.177', 'RG 271.27', 'RG 271.52', 'RG 271.170', 'RG 271.101', 'RG 271.165']",requirement
7,8,0.557616,idr_process_response,Complaint management documentation requirements,['RG 271.172'],requirement
8,9,0.567427,idr_process_response,Internal complaint management procedure,"['RG 271.175', 'RG 271.176']",requirement
9,10,0.590179,idr_process_response,Training for Complaint Handling Staff,['RG 271.150'],requirement


In [20]:
for _, row in deduped_retrieval_df.head(8).iterrows():
    print(f"\n--- Final Retrieved Chunk {row['final_rank']} ---")
    print("Distance:", row["distance"])
    print("Matched query:", row["query_name"])
    print("Title:", row["title"])
    print("RG refs:", row["rg_refs"])
    print("Type:", row["chunk_type"])
    print(row["text"][:1500])


--- Final Retrieved Chunk 1 ---
Distance: 0.35919591784477234
Matched query: idr_timeframes
Title: Overview of maximum IDR timeframes and response requirements
RG refs: ['RG 271.49', 'RG 271.50']
Type: overview
This section sets out: • when financial firms should acknowledge a complaint; • what financial firms must include in an IDR response; • the maximum timeframes that financial firms have to provide an IDR response; and • when a financial firm does not have to provide an IDR response within the maximum IDR timeframe. We also set out our expectations about how firms’ IDR processes will interact with AFCA. Timeliness is central to effective complaint management and is a key performance measure of a firm’s IDR process. Findings from ASIC’s research into the consumer experience of the IDR journey indicate that delays and frictions in the IDR process can create real barriers for consumers and damage the consumer–firm relationship. Note: See Report 603 The consumer journey through the I

In [21]:
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

multi_retrieval_df.to_csv(
    REPORTS_DIR / "multi_query_retrieval_raw_results.csv",
    index=False
)

deduped_retrieval_df.to_csv(
    REPORTS_DIR / "multi_query_retrieval_deduped_results.csv",
    index=False
)

print("Multi-query retrieval results saved to reports/ folder.")

Multi-query retrieval results saved to reports/ folder.


## RAG Complaint Handling Note Generation

This section uses the retrieved RG 271 chunks as context for a short complaint handling note.

The goal is not to provide legal advice, but to demonstrate how retrieved regulatory guidance can support complaint triage and internal handling.

The generated output should:
- identify the likely complaint handling context,
- reference relevant RG 271 guidance,
- suggest internal handling considerations,
- flag that human review is required for final decisions.

In [22]:
top_context_df = deduped_retrieval_df.head(5).copy()

rag_context = "\n\n".join([
    f"Title: {row['title']}\nRG refs: {row['rg_refs']}\nText: {row['text']}"
    for _, row in top_context_df.iterrows()
])

print(rag_context[:3000])

Title: Overview of maximum IDR timeframes and response requirements
RG refs: ['RG 271.49', 'RG 271.50']
Text: This section sets out: • when financial firms should acknowledge a complaint; • what financial firms must include in an IDR response; • the maximum timeframes that financial firms have to provide an IDR response; and • when a financial firm does not have to provide an IDR response within the maximum IDR timeframe. We also set out our expectations about how firms’ IDR processes will interact with AFCA. Timeliness is central to effective complaint management and is a key performance measure of a firm’s IDR process. Findings from ASIC’s research into the consumer experience of the IDR journey indicate that delays and frictions in the IDR process can create real barriers for consumers and damage the consumer–firm relationship. Note: See Report 603 The consumer journey through the Internal Dispute Resolution process of financial service providers (REP 603). Important measures of tim

In [23]:
sample_complaint = complaints_df.loc[0, "redacted_text"]

print(sample_complaint[:1500])

This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .

After my original CFPB complaint, the collector responded with additional materials that raise further concerns rather than resolving the dispute.

Among other issues : The collectors own documents appear to reflect mailing to <PERSON> rather than my actual mailing designation of <PERSON>, raising substantial concerns regarding lawful notice and delivery.

The collector continues attempting to collect large storage-related charges that escalated far beyond the apparent value of the vehicle, despite prior attempts to reasonably resolve the matter.

The collector has failed to provide competent documentation establishing a commercially reasonable lien sale process, including meaningful documentation regarding sale procedures, advertising, bidding, or disposition methodology.

The collector also failed to provide contemporaneous storage reco

In [24]:
RAG_SYSTEM_PROMPT = """
You are a complaint intelligence assistant for a financial services setting.

You must use only the provided RG 271 context.
Do not provide legal advice.
Do not invent regulatory references.
If the context is insufficient, say that human review is required.

Write a short internal complaint handling note with:
1. Complaint context
2. Relevant RG 271 guidance
3. Suggested handling considerations
4. Human review flag
"""

In [26]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pathlib import Path

# Jupyter kernels often run from a different directory than the notebook file.
# Walk upward from the kernel cwd so complaint-intelligence-platform/.env is found.
env_path = None
for base in [Path.cwd(), *Path.cwd().parents]:
    candidate = base / ".env"
    if candidate.exists():
        env_path = candidate
        break

if env_path:
    load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        f"OPENAI_API_KEY not found. Checked current working directory chain starting from: {Path.cwd()}"
    )

client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Complaint narrative:
{sample_complaint}

Retrieved RG 271 context:
{rag_context}

Generate the internal complaint handling note.
"""
        }
    ]
)

rag_note = response.choices[0].message.content

print("Loaded .env from:", env_path)
print(rag_note)


Loaded .env from: /Users/osmanorka/Complaint-Intelligence-Platform-PII-Safe-Classification-RAG-Assistant-for-Financial-Complaints/complaint-intelligence-platform/.env
Internal Complaint Handling Note

1. Complaint Context:
The complainant disputes a towing/storage deficiency balance allegedly owed to a third party. They raise concerns about improper notice due to mailing errors, lack of competent documentation supporting the lien sale process, absence of contemporaneous storage records and accounting, and unsubstantiated interest charges. Despite prior complaints and attempts to resolve, the collector continues collection efforts and asserts the validity of the disputed balance.

2. Relevant RG 271 Guidance:
- RG 271.49, RG 271.50: Emphasize timely acknowledgement and comprehensive IDR responses to complaints.
- RG 271.63: Recognizes complexity and third-party information availability may affect response times but encourages firms to meet or exceed maximum IDR timeframes.
- RG 271.90: 

In [27]:
RAG_SYSTEM_PROMPT = """
You are a complaint intelligence assistant for a financial services setting.

You must use only the provided RG 271 context.
Do not provide legal advice.
Do not invent regulatory references.
If the context is insufficient, say that human review is required.

Write a short internal complaint handling note with:
1. Complaint context
2. Relevant RG 271 guidance
3. Suggested handling considerations
4. Human review flag

Keep the tone professional, concise, and suitable for an internal complaints team.
"""

In [28]:
def build_retrieval_queries(complaint_text: str) -> list[dict]:
    """
    Build multiple focused retrieval queries for a complaint.
    
    Instead of sending only the raw complaint to Chroma, we search from
    several regulatory angles: complaint definition, IDR process, timeframes,
    AFCA escalation, and dispute context.
    """
    
    return [
        {
            "query_name": "complaint_definition",
            "query_text": f"""
            Regulatory guidance on what counts as a complaint, expression of dissatisfaction,
            complainant, disputed transaction, and complaint handling.

            Complaint:
            {complaint_text[:1200]}
            """
        },
        {
            "query_name": "idr_process_response",
            "query_text": f"""
            Regulatory guidance on internal dispute resolution process, IDR response,
            complaint handling obligations, written response, reasons for decision,
            and financial firm responsibilities.

            Complaint:
            {complaint_text[:1200]}
            """
        },
        {
            "query_name": "idr_timeframes",
            "query_text": f"""
            Regulatory guidance on maximum IDR timeframes, acknowledgement of complaint,
            response time limits, complaint resolution timeframe, AFCA referral,
            and delay in complaint handling.

            Complaint:
            {complaint_text[:1200]}
            """
        },
        {
            "query_name": "dispute_context",
            "query_text": f"""
            Consumer complaint involving disputed charges, disputed debt, account issue,
            transaction dispute, collection activity, financial hardship, enforcement,
            or unresolved complaint handling.

            Complaint:
            {complaint_text[:1200]}
            """
        }
    ]


print("Retrieval query builder ready.")

Retrieval query builder ready.


In [29]:
def retrieve_rg271_context(
    complaint_text: str,
    collection,
    embedding_model,
    n_results_per_query: int = 5,
    top_k_final: int = 5
) -> pd.DataFrame:
    """
    Retrieve relevant RG 271 chunks using multi-query retrieval.
    
    Returns a deduplicated dataframe of the top retrieved chunks.
    """
    
    queries = build_retrieval_queries(complaint_text)
    all_results = []
    
    for item in queries:
        query_embedding = embedding_model.encode(
            item["query_text"],
            normalize_embeddings=True
        ).tolist()
        
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results_per_query
        )
        
        for i in range(len(results["ids"][0])):
            all_results.append({
                "query_name": item["query_name"],
                "rank_within_query": i + 1,
                "id": results["ids"][0][i],
                "distance": results["distances"][0][i],
                "title": results["metadatas"][0][i].get("title"),
                "rg_refs": results["metadatas"][0][i].get("rg_refs"),
                "chunk_type": results["metadatas"][0][i].get("chunk_type"),
                "text": results["documents"][0][i]
            })
    
    retrieval_df = pd.DataFrame(all_results)
    
    deduped_df = (
        retrieval_df
        .sort_values("distance", ascending=True)
        .groupby("id", as_index=False)
        .agg({
            "distance": "min",
            "query_name": lambda x: ", ".join(sorted(set(x))),
            "title": "first",
            "rg_refs": "first",
            "chunk_type": "first",
            "text": "first"
        })
        .sort_values("distance", ascending=True)
        .head(top_k_final)
        .reset_index(drop=True)
    )
    
    deduped_df["final_rank"] = deduped_df.index + 1
    
    return deduped_df


print("RG 271 retrieval function ready.")

RG 271 retrieval function ready.


In [30]:
def build_rag_context(retrieved_df: pd.DataFrame) -> str:
    """
    Convert retrieved RG 271 chunks into a structured context string for the LLM.
    """
    
    context_parts = []
    
    for _, row in retrieved_df.iterrows():
        context_parts.append(
            f"""
Title: {row['title']}
RG refs: {row['rg_refs']}
Chunk type: {row['chunk_type']}
Text:
{row['text']}
"""
        )
    
    return "\n\n---\n\n".join(context_parts)


print("RAG context builder ready.")

RAG context builder ready.


In [31]:
def generate_complaint_handling_note(
    complaint_text: str,
    collection,
    embedding_model,
    client,
    model: str = "gpt-4.1-mini",
    top_k_context: int = 5
) -> dict:
    """
    Generate an internal complaint handling note using retrieved RG 271 context.
    
    Returns:
        {
            "complaint_text": original complaint text,
            "retrieved_context": dataframe of retrieved chunks,
            "rag_context": context string,
            "handling_note": generated note
        }
    """
    
    retrieved_df = retrieve_rg271_context(
        complaint_text=complaint_text,
        collection=collection,
        embedding_model=embedding_model,
        n_results_per_query=5,
        top_k_final=top_k_context
    )
    
    rag_context = build_rag_context(retrieved_df)
    
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": RAG_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
Complaint narrative:
{complaint_text}

Retrieved RG 271 context:
{rag_context}

Generate the internal complaint handling note.
"""
            }
        ]
    )
    
    handling_note = response.choices[0].message.content
    
    return {
        "complaint_text": complaint_text,
        "retrieved_context": retrieved_df,
        "rag_context": rag_context,
        "handling_note": handling_note
    }


print("Complaint handling note generator ready.")

Complaint handling note generator ready.


In [32]:
sample_complaint = complaints_df.loc[0, "redacted_text"]

result = generate_complaint_handling_note(
    complaint_text=sample_complaint,
    collection=collection,
    embedding_model=embedding_model,
    client=client,
    model="gpt-4.1-mini",
    top_k_context=5
)

print(result["handling_note"])

Internal Complaint Handling Note

1. Complaint Context:
The complainant disputes a towing/storage deficiency balance allegedly owed to a third party. They raise concerns about improper notice due to mailing errors, lack of adequate documentation supporting the debt and lien sale process, absence of contemporaneous storage records and accounting, and unsubstantiated interest charges. The complainant asserts the debt is disputed in full and that collection efforts continue despite unresolved issues.

2. Relevant RG 271 Guidance:
- RG 271.30: The complainant’s expression of dissatisfaction and dispute triggers the firm’s obligation to handle the matter as a complaint under Internal Dispute Resolution (IDR) requirements.
- RG 271.53 and RG 271.84(a): The firm must provide a comprehensive IDR response addressing the complaint’s substance, including reasons for any rejection or partial rejection, and inform the complainant of their right to escalate to AFCA.
- RG 271.90: Enforcement action s

In [33]:
result["retrieved_context"][[
    "final_rank",
    "distance",
    "query_name",
    "title",
    "rg_refs",
    "chunk_type"
]]

,final_rank,distance,query_name,title,rg_refs,chunk_type
0,1,0.731525,"complaint_definition, dispute_context, idr_process_response",Clarification of what is not a complaint,['RG 271.33'],definition
1,2,0.766016,"complaint_definition, dispute_context, idr_timeframes",Complaints Involving Hardship Notices and Enforcement Postponement,['RG 271.116'],requirement
2,3,0.776976,"complaint_definition, dispute_context, idr_process_response, idr_timeframes",Proactive approach to identifying complaints,['RG 271.30'],requirement
3,4,0.795850,idr_timeframes,Contents of an IDR response,"['RG 271.53', 'RG 271.84(a)']",requirement
4,5,0.804403,idr_timeframes,Purpose of enforcement action restrictions and reasonable time period,['RG 271.90'],requirement


In [34]:
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Save retrieved context
result["retrieved_context"].to_csv(
    REPORTS_DIR / "sample_rag_retrieved_context.csv",
    index=False
)

# Save generated note
with open(REPORTS_DIR / "sample_rag_complaint_handling_note.txt", "w", encoding="utf-8") as f:
    f.write(result["handling_note"])

print("Sample RAG result saved to reports/ folder.")

Sample RAG result saved to reports/ folder.


## Batch RAG Demo on Multiple Complaints

This section tests the RAG complaint handling workflow on multiple complaint examples.

The goal is to check whether the system can:
- retrieve relevant RG 271 context for different complaint narratives,
- generate a useful internal complaint handling note,
- preserve human review as the final decision point,
- produce auditable outputs for each example.

In [ ]:
complaints_df["label"].value_counts()

In [36]:
demo_labels = [
    "Debt collection",
    "Credit card",
    "Checking or savings account"
]

demo_complaints = []

for label in demo_labels:
    sample_row = complaints_df[complaints_df["label"] == label].sample(
        1,
        random_state=42
    ).iloc[0]
    
    demo_complaints.append({
        "demo_id": f"demo_{len(demo_complaints) + 1}",
        "label": label,
        "complaint_text": sample_row["redacted_text"]
    })

demo_complaints_df = pd.DataFrame(demo_complaints)

demo_complaints_df[["demo_id", "label", "complaint_text"]]

,demo_id,label,complaint_text
0,demo_1,Debt collection,"I request validation of the alleged debt referenced in your recent communication, including the name of the original creditor, the total amount owed, documentation establishing my responsibility for the debt, and proof that your company is authorized to collect it."
1,demo_2,Credit card,On <LOCATION>/XX/year> I visited the Wells Fargo branch in <PERSON> <LOCATION>. I went into to close my credit card account. I did not request a credit card. Wells Fargo offered to fund the repair of a septic tank. I did not know until after the funding that it was a credit card. When I paid off...
2,demo_3,Checking or savings account,"<DATE_TIME> of XXXX, I went with my <PERSON> stepfather to our local Bank of American office <PERSON> XXXX XXXX <LOCATION> where he opened a trust account for his newly signed Revocable Trust. I am not sure exactly how much money he deposited into this account from his existing account with the ..."


In [37]:
batch_rag_results = []

for item in demo_complaints:
    print(f"Processing {item['demo_id']} — {item['label']}...")
    
    result = generate_complaint_handling_note(
        complaint_text=item["complaint_text"],
        collection=collection,
        embedding_model=embedding_model,
        client=client,
        model="gpt-4.1-mini",
        top_k_context=5
    )
    
    batch_rag_results.append({
        "demo_id": item["demo_id"],
        "label": item["label"],
        "complaint_text": item["complaint_text"],
        "handling_note": result["handling_note"],
        "retrieved_context": result["retrieved_context"]
    })
    
    print("Done.")

print("Batch RAG demo completed.")

Processing demo_1 — Debt collection...
Done.
Processing demo_2 — Credit card...
Done.
Processing demo_3 — Checking or savings account...
Done.
Batch RAG demo completed.


In [38]:
batch_summary_rows = []

for item in batch_rag_results:
    retrieved_df = item["retrieved_context"]
    
    batch_summary_rows.append({
        "demo_id": item["demo_id"],
        "label": item["label"],
        "top_rg_refs": "; ".join(retrieved_df["rg_refs"].astype(str).head(3).tolist()),
        "top_titles": " | ".join(retrieved_df["title"].astype(str).head(3).tolist()),
        "note_chars": len(item["handling_note"])
    })

batch_rag_summary_df = pd.DataFrame(batch_summary_rows)

batch_rag_summary_df

,demo_id,label,top_rg_refs,top_titles,note_chars
0,demo_1,Debt collection,"['RG 271.49', 'RG 271.50']; ['RG 271.90']; ['RG 271.22', 'RG 271.106']",Overview of maximum IDR timeframes and response requirements | Purpose of enforcement action restrictions and reasonable time period | Contents of the Guide Regarding IDR and AFCA Interaction,1463
1,demo_2,Credit card,"['RG 271.91']; ['RG 271.53', 'RG 271.84(a)']; ['RG 271.55']",Exceptions and procedures for credit-related hardship complaints | Contents of an IDR response | Level of detail and privacy considerations in IDR responses,1884
2,demo_3,Checking or savings account,['RG 271.59']; ['RG 271.77']; ['RG 271.76'],Different timeframes for traditional trustee complaints | Suspension of IDR timeframe during legal proceedings or court directions | Traditional trustee obligations during 45 calendar day IDR timeframe,1775


In [39]:
for item in batch_rag_results:
    print("\n" + "="*80)
    print(f"{item['demo_id']} — {item['label']}")
    print("="*80)
    print(item["handling_note"])


demo_1 — Debt collection
Internal Complaint Handling Note

1. Complaint Context:
The complainant requests validation of an alleged debt, specifically seeking the original creditor’s name, total amount owed, documentation proving their responsibility for the debt, and evidence that the company is authorized to collect it.

2. Relevant RG 271 Guidance:
- RG 271.49 and RG 271.50 emphasize timely acknowledgement and comprehensive IDR responses to complaints.
- RG 271.86 requires an IDR response within 21 calendar days for complaints involving default notices.
- RG 271.90 highlights the importance of dealing with complaints genuinely within a reasonable timeframe to allow escalation to AFCA if unresolved.
- RG 271.111 mandates informing complainants of their right to escalate unresolved complaints to AFCA and providing access details.

3. Suggested Handling Considerations:
- Acknowledge the complaint promptly.
- Provide a detailed IDR response within 21 calendar days, including all request

In [40]:
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

batch_rag_summary_df.to_csv(
    REPORTS_DIR / "batch_rag_demo_summary.csv",
    index=False
)

# Her demo için note + retrieved context kaydet
for item in batch_rag_results:
    demo_id = item["demo_id"]
    
    with open(REPORTS_DIR / f"{demo_id}_rag_handling_note.txt", "w", encoding="utf-8") as f:
        f.write(item["handling_note"])
    
    item["retrieved_context"].to_csv(
        REPORTS_DIR / f"{demo_id}_retrieved_context.csv",
        index=False
    )

print("Batch RAG demo outputs saved to reports/ folder.")

Batch RAG demo outputs saved to reports/ folder.
